# 09 · LightGBM re-ranker (Direction 3)

**Goal:** re-rank each candidate list with LightGBM on the feature table of `08_features` (out-of-fold scores of two cross-encoders, DeBERTa-base and DeBERTa-large ce4, + within-query, lexical, bi-encoder, brand-list, product and query features), and measure what it adds over DeBERTa-large alone and over a plain score blend.

First version (DeBERTa-base only, 48 features): runs `d3_lgbm_*` and files `d3_grid.csv`, `d3_ablation.csv`, `d3_feature_importance.csv` in `data/results/`. This version adds the large model (groups `ce3`, `ce3_list`; 62 features) and writes new names (`d3_lgbm_ce3_*`, `d3_ce3_*.csv`), so the old results stay.

**Design.**
- **Cross-validation on train** with the same 5 folds as 07 (`fold` column). The CE features of a train pair come from the fold model that never saw its query, so a LightGBM fold model never sees a CE score fitted on its own held-out queries. Early stopping uses the held-out fold, so the CV score is slightly optimistic; **val is never used for fitting or stopping** and gives the clean comparison.
- **Val / test scores** = mean of the 5 fold models (like 07), test saved blind for `10_report`.
- **Objectives:** `lambdarank` (label = I 0, C 1, S 2, E 3, `label_gain` = official gains 0 / 0.01 / 0.1 / 1, full-list nDCG) vs `multiclass` 4 classes → expected gain. Each with a small grid (`GRID`); the setting with the best **CV nDCG** is kept per objective, and the objective with the best CV nDCG goes to the ablation.
- **References** (paired bootstrap on the same queries): `d3_ce_debertalarge_main_ce4` = the large CE score alone (train OOF 0.8923, val 0.8970); `d3_lgbm_multiclass` = the first LightGBM (0.8884, 0.8947); `d3_blend_lgbm_large` = within-query z-scores of these two, weight chosen on train OOF (about 0.8949, 0.8992).
- **Rule, fixed before the run:** the new LightGBM is kept only if it beats the blend on CV (CI excludes 0) and is not worse on val; otherwise the blend is the better system. Test is not looked at for this choice.
- **Ablation** (best objective and setting): drop one feature group at a time, plus `no_base_ce` / `no_large_ce` (all features of one CE), `ce_only` (the two CEs only) and `no_ce_any` (no CE feature at all). `no_large_ce` is the feature set of the first version.

**Kaggle settings:** CPU, no internet (LightGBM is preinstalled). `smoke` ~5 min; `full` ~1–1.5 h (about 95 LightGBM fits; a 5-fold run of the first version took 1–4 min).

**Inputs** (Add Input):
- Dataset `wm-src` with `data/results/scores/d3_lgbm_multiclass_*` (the version used for 12). Required.
- Notebook Output **08_features**, the version with the `ce3` groups (`features.parquet`, `feature_groups.json`). Required; remove the old version from the inputs.

**How to run.** `MODE = 'smoke'` (10% of the queries, one setting, 200 rounds) → check the verdict at the end of section 3 → `MODE = 'full'`, **Save Version → Save & Run All**.

**Output** (`/kaggle/working/results/`): `d3_ce_debertalarge_main_ce4.json`, `d3_blend_lgbm_large.json`, `d3_lgbm_ce3_lambdarank.json`, `d3_lgbm_ce3_multiclass.json` + `scores/<run>_{oof,val,test}.parquet` (test blind); `d3_ce3_grid.csv`, `d3_ce3_ablation.csv`, `d3_ce3_feature_importance.csv`.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import json, os, time

import lightgbm as lgb
import numpy as np
import pandas as pd

from src.data import find
from src.evaluation import RunLog, paired_bootstrap, summarize
from src.features import within_query

MODE = 'smoke'    # 'smoke': 10% of the queries, one setting, 200 rounds | 'full'
ABLATION = True   # full mode: run section 5 (13 variants, 65 fits)

INPUT = '/kaggle/input'
RES = Path('/kaggle/working/results')
N_FOLDS, SEED = 5, 42
LABEL = {'I': 0, 'C': 1, 'S': 2, 'E': 3}
GAIN = np.array([0.0, 0.01, 0.1, 1.0])          # official gain of label code 0..3
REF = 'd3_ce_debertalarge_main_ce4'             # the large CE score alone
OLD = 'd3_lgbm_multiclass'                      # first version (DeBERTa-base only), from wm-src
BLEND = 'd3_blend_lgbm_large'                   # z(OLD) and z(large CE), weight chosen on OOF
NEW = 'd3_lgbm_ce3'                             # runs of this notebook: NEW_lambdarank, NEW_multiclass

BASE = {'learning_rate': 0.05, 'num_leaves': 31, 'min_data_in_leaf': 100, 'feature_fraction': 0.8,
        'lambda_l2': 1.0, 'seed': SEED, 'deterministic': True, 'force_row_wise': True,
        'num_threads': os.cpu_count(), 'verbose': -1}
GRID = {'small': {'num_leaves': 15, 'min_data_in_leaf': 200},
        'base': {},
        'large': {'num_leaves': 63, 'min_data_in_leaf': 50}}
MAX_ROUNDS, PATIENCE, SAMPLE = 3000, 100, None

assert MODE in ('smoke', 'full')
if MODE == 'smoke':
    GRID, MAX_ROUNDS, PATIENCE, SAMPLE = {'base': {}}, 200, 20, 0.10
PREFIX = 'smoke_' if MODE == 'smoke' else ''
print('lightgbm', lgb.__version__, '| MODE', MODE, '| cpus', os.cpu_count())

## 1. Data

Rows stay sorted by `query_id` (as written by 08), so every query is one contiguous block: LightGBM `group` sizes are taken in that order.

In [ ]:
feat_hint = 'Add Input -> Notebook Output -> 08_features'
df = pd.read_parquet(find('features.parquet', feat_hint, INPUT))
GROUPS = json.loads(find('feature_groups.json', feat_hint, INPUT).read_text())['groups']
assert 'ce3' in GROUPS, 'features.parquet is from the first version of 08: attach the version with the ce3 groups and remove the old one'
FEATURES = [c for cols in GROUPS.values() for c in cols]
df['y'] = df['esci_label'].map(LABEL).astype(int)

if SAMPLE:
    q = df[['query_id', 'split']].drop_duplicates().groupby('split', group_keys=False).sample(frac=SAMPLE, random_state=SEED)
    df = df[df['query_id'].isin(q['query_id'])]
data = {s: df[df['split'] == s].sort_values(['query_id', 'example_id']).reset_index(drop=True) for s in ['train', 'val', 'test']}
train, val, test = data['train'], data['val'], data['test']
del df

assert set(train['fold']) == set(range(N_FOLDS)) and (val['fold'] == -1).all() and (test['fold'] == -1).all()
assert train.groupby('query_id')['fold'].nunique().max() == 1
EVAL_AT = int(max(d.groupby('query_id').size().max() for d in data.values()))   # full-list nDCG
print(len(FEATURES), 'features |', {s: f"{d['query_id'].nunique():,} q / {len(d):,} pairs" for s, d in data.items()},
      '| longest list', EVAL_AT)

OBJ = {'lambdarank': {'objective': 'lambdarank', 'metric': 'ndcg', 'eval_at': [EVAL_AT], 'label_gain': GAIN.tolist(),
                      'lambdarank_truncation_level': EVAL_AT},
       'multiclass': {'objective': 'multiclass', 'num_class': 4, 'metric': 'multi_logloss'}}

## 2. References: large CE alone, first LightGBM, blend

The large CE score of 08: out-of-fold on train, mean of the 5 fold models on val. Test is not logged here (it is the assembled file of 12). The first LightGBM is read from `wm-src`. The blend is `(1 - w) z(LightGBM) + w z(large CE)` with z-scores inside each query; `w` is the best of 0.1–0.9 on train OOF, val is only reported.

In [ ]:
parts = {'oof': train, 'val': val, 'test': test}
log = RunLog(RES, {s: p[['example_id', 'query_id', 'esci_label']] for s, p in parts.items()})
REF, BLEND, NEW = PREFIX + REF, PREFIX + BLEND, PREFIX + NEW
log.log(REF, {'oof': train['ce3_score'], 'val': val['ce3_score']},
        {'mode': MODE, 'model': 'DeBERTa-large ce score alone (12 assemble, via 08_features)', 'sample': SAMPLE})
assert log.load(OLD, splits=['oof', 'val'], root=INPUT), f'{OLD} scores not found: attach wm-src with data/results/scores'


def old_z(part, split):   # z-score inside the query of the first LightGBM's scores
    sc = pd.read_parquet(find(f'{OLD}_{split}.parquet', 'Add Input -> wm-src', INPUT))[['example_id', 'score']]
    d = part[['example_id', 'query_id']].merge(sc, on='example_id', how='left', validate='1:1')
    assert d['score'].notna().all(), f'{OLD} does not cover {split}'
    return within_query(d, 'score')['score_z'].to_numpy()


z_old = {s: old_z(p, s) for s, p in parts.items()}


def blend(split, w):
    return (1 - w) * z_old[split] + w * parts[split]['ce3_score_z'].to_numpy()


weights = pd.DataFrame([{'w_large': w, 'oof_ndcg': summarize(train, blend('oof', w))[0]['ndcg'],
                         'val_ndcg': summarize(val, blend('val', w))[0]['ndcg']}
                        for w in np.round(np.arange(0.1, 1.0, 0.1), 1)])
display(weights)
W = float(weights['w_large'][weights['oof_ndcg'].idxmax()])
log.log(BLEND, {s: blend(s, W) for s in parts},
        {'mode': MODE, 'model': f'{1 - W:.1f} z({OLD}) + {W:.1f} z(large CE)', 'w_large': W, 'weight_chosen_on': 'oof',
         'sample': SAMPLE}, compare_to=REF, blind=('test',))

## 3. Fit with 5-fold CV

`fit_cv` trains one model per fold on the 4 other folds, stops on the held-out fold, predicts the held-out fold (`oof`) and val / test (mean of the 5 models). Multiclass scores are expected gains; lambdarank scores are raw model scores (averaging them over the folds keeps the order of a single model's scale, as the 5 models are trained alike).

In [ ]:
def dataset(d, feats, obj, ref=None):
    group = d.groupby('query_id', sort=False).size().to_numpy() if obj == 'lambdarank' else None
    return lgb.Dataset(d[feats], label=d['y'], group=group, reference=ref, free_raw_data=False)


def score(model, d, feats, obj):
    p = model.predict(d[feats], num_iteration=model.best_iteration)
    return p @ GAIN if obj == 'multiclass' else p


def fit_cv(obj, feats, params):
    t0 = time.time()
    out = {'oof': np.zeros(len(train)), 'val': np.zeros(len(val)), 'test': np.zeros(len(test)),
           'iters': [], 'importance': np.zeros(len(feats))}
    fold = train['fold'].to_numpy()
    for k in range(N_FOLDS):
        tr, ho = train[fold != k], train[fold == k]
        dtr = dataset(tr, feats, obj)
        model = lgb.train({**BASE, **OBJ[obj], **params}, dtr, MAX_ROUNDS,
                          valid_sets=[dataset(ho, feats, obj, ref=dtr)],
                          callbacks=[lgb.early_stopping(PATIENCE, verbose=False)])
        out['oof'][fold == k] = score(model, ho, feats, obj)
        for s, d in (('val', val), ('test', test)):
            out[s] += score(model, d, feats, obj) / N_FOLDS
        out['iters'].append(int(model.best_iteration))
        out['importance'] += model.feature_importance('gain') / N_FOLDS
    out['minutes'] = round((time.time() - t0) / 60, 1)
    return out


def metrics(out):
    return {s: summarize(d, out[s])[0] for s, d in (('oof', train), ('val', val))}

In [ ]:
fits, rows = {}, []
for obj in OBJ:
    for cfg, params in GRID.items():
        out = fit_cv(obj, FEATURES, params)
        m = metrics(out)
        fits[obj, cfg] = out
        rows.append({'objective': obj, 'setting': cfg, 'cv_ndcg': m['oof']['ndcg'], 'val_ndcg': m['val']['ndcg'],
                     'val_ndcg10': m['val']['ndcg10'], 'val_e_over_s': m['val']['e_over_s'],
                     'best_iters': out['iters'], 'minutes': out['minutes']})
        print(rows[-1])
grid = pd.DataFrame(rows)
display(grid)

best_cfg = {obj: grid[grid['objective'] == obj].sort_values('cv_ndcg', ascending=False)['setting'].iloc[0] for obj in OBJ}
BEST_OBJ = max(OBJ, key=lambda o: grid.set_index(['objective', 'setting']).loc[(o, best_cfg[o]), 'cv_ndcg'])
print('setting chosen by CV nDCG:', best_cfg, '| objective for the ablation:', BEST_OBJ)

In [ ]:
if MODE == 'smoke':
    cv = grid.groupby('objective')['cv_ndcg'].max()
    checks = {'5 fold models per fit': all(len(i) == N_FOLDS for i in grid['best_iters']),
              'scores finite': all(np.isfinite(f[s]).all() for f in fits.values() for s in ('oof', 'val', 'test')),
              'lambdarank CV above random (0.742)': cv['lambdarank'] > 0.742,
              'multiclass CV above random (0.742)': cv['multiclass'] > 0.742,
              'val above random (0.742)': (grid['val_ndcg'] > 0.742).all()}
    for c, ok in checks.items():
        print(f"{'OK   ' if ok else 'CHECK'} {c}")
    print('minutes per 5-fold fit:', grid['minutes'].tolist(), '(full data: about x10 rows, up to x15 rounds)')
    print('Smoke test passed: switch MODE to full.' if all(checks.values()) else 'Something failed: send me this output.')

## 4. Runs: best setting per objective vs the references

Logged with `RunLog` (oof + val scored, test blind) and compared with the large CE alone on the same queries (paired bootstrap). The lines below compare with the first LightGBM and with the blend (the rule at the top), and the two objectives with each other.

In [ ]:
for obj in OBJ:
    cfg, out = best_cfg[obj], fits[obj, best_cfg[obj]]
    log.log(f'{NEW}_{obj}', {s: out[s] for s in ('oof', 'val', 'test')},
            {'mode': MODE, 'objective': obj, 'setting': cfg, 'params': {**BASE, **OBJ[obj], **GRID[cfg]},
             'features': FEATURES, 'n_features': len(FEATURES), 'n_folds': N_FOLDS, 'best_iterations': out['iters'],
             'max_rounds': MAX_ROUNDS, 'patience': PATIENCE, 'sample': SAMPLE, 'fit_minutes': out['minutes'],
             'val_test_scores': 'mean of the 5 fold models'},
            compare_to=REF, blind=('test',))
display(log.table(('ndcg', 'ndcg_ci95', 'ndcg10', 'mrr_e', 'e_over_s', 'n_queries')))
log.print_paired()
a, b = f'{NEW}_multiclass', f'{NEW}_lambdarank'
for s in ('oof', 'val'):
    for x, y in [(OLD, a), (OLD, b), (BLEND, a), (BLEND, b), (a, b)]:
        print(f'{s:4s} {y} - {x}:', paired_bootstrap(log.per_query[x][s], log.per_query[y][s]))

## 5. Ablation and feature importance

Best objective and setting; each variant is a full 5-fold fit. Differences are paired with the all-features model on the same queries (CV on train, and val). Scores of the variants are not saved, only this table.

In [ ]:
cfg = best_cfg[BEST_OBJ]
full = fits[BEST_OBJ, cfg]
pq_full = {s: summarize(d, full[s])[1] for s, d in (('oof', train), ('val', val))}

imp = pd.DataFrame({'feature': FEATURES, 'group': [g for g, cols in GROUPS.items() for _ in cols],
                    'gain': full['importance']})
imp['share'] = (imp['gain'] / imp['gain'].sum()).round(4)
imp = imp.sort_values('gain', ascending=False).reset_index(drop=True)
display(imp.head(20))
display(imp.groupby('group')['share'].sum().sort_values(ascending=False).round(3))

variants = {f'no_{g}': [c for c in FEATURES if c not in GROUPS[g]] for g in GROUPS}
base_ce, large_ce = GROUPS['ce'] + GROUPS['ce_list'], GROUPS['ce3'] + GROUPS['ce3_list']
variants['no_base_ce'] = [c for c in FEATURES if c not in base_ce]
variants['no_large_ce'] = [c for c in FEATURES if c not in large_ce]   # feature set of the first version
variants['ce_only'] = base_ce + large_ce
variants['no_ce_any'] = [c for c in FEATURES if c not in base_ce + large_ce]
rows = []
for name, feats in (variants.items() if ABLATION else []):
    out = fit_cv(BEST_OBJ, feats, GRID[cfg])
    r = {'variant': name, 'n_features': len(feats), 'minutes': out['minutes']}
    for s, d in (('oof', train), ('val', val)):
        m, pq = summarize(d, out[s])
        p = paired_bootstrap(pq_full[s], pq)
        r.update({f'{s}_ndcg': m['ndcg'], f'{s}_diff': p['diff'], f'{s}_ci95': p['ci95']})
    rows.append(r)
    print(r)
ablation = pd.DataFrame(rows)
if len(ablation):
    display(ablation.sort_values('val_diff'))

## 6. Save

Run jsons and scores were written by `RunLog` in section 4. Download `results/` into `data/results/` (about 10 MB per LightGBM run).

In [ ]:
grid.to_csv(RES / f'{PREFIX}d3_ce3_grid.csv', index=False)
imp.to_csv(RES / f'{PREFIX}d3_ce3_feature_importance.csv', index=False)
if len(ablation):
    ablation.to_csv(RES / f'{PREFIX}d3_ce3_ablation.csv', index=False)
print(sorted(p.name for p in RES.glob('*')))